# SQL Query Agent

Run these cells from top to bottom to understand the demo database, connect the SQL agent, and ask questions in plain English.

## 1. Import the agent helpers

In [ ]:
import sqlite3
from pathlib import Path

from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI

agent_folder = Path.cwd().parent
print("Notebook dependencies imported.")

SQL agent helpers imported.


In [ ]:
def create_demo_database(path: Path) -> Path:
    """Create a small e-commerce database for notebook experiments."""

    with sqlite3.connect(path) as connection:
        connection.executescript("""
            CREATE TABLE IF NOT EXISTS customers (id INTEGER PRIMARY KEY, name TEXT, country TEXT);
            CREATE TABLE IF NOT EXISTS products (id INTEGER PRIMARY KEY, name TEXT, price REAL);
            CREATE TABLE IF NOT EXISTS orders (id INTEGER PRIMARY KEY, customer_id INTEGER, product_id INTEGER, quantity INTEGER, total REAL);
            INSERT OR IGNORE INTO customers VALUES (1, 'Alice', 'USA'), (2, 'Bob', 'UK'), (3, 'Carlos', 'Brazil'), (4, 'Diana', 'USA');
            INSERT OR IGNORE INTO products VALUES (1, 'Laptop Pro', 1299.99), (2, 'Wireless Mouse', 29.99), (3, 'Python Book', 49.99);
            INSERT OR IGNORE INTO orders VALUES (1, 1, 1, 1, 1299.99), (2, 1, 2, 2, 59.98), (3, 2, 3, 1, 49.99), (4, 3, 1, 1, 1299.99);
        """)
    return path

In [ ]:
database_path = agent_folder / "demo_notebook.sqlite"
create_demo_database(database_path)
print(f"Created: {database_path.name}")

In [ ]:
def build_agent(path: Path):
    """Build a read-only SQL agent for the demo database."""

    database = SQLDatabase.from_uri(f"sqlite:///file:{path.as_posix()}?mode=ro&uri=true")
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    toolkit = SQLDatabaseToolkit(db=database, llm=model)
    sql_agent = create_sql_agent(llm=model, toolkit=toolkit, agent_type="openai-tools", verbose=False)
    return sql_agent, database

In [ ]:
agent, database = build_agent(database_path)
print(database.get_usable_table_names())

In [ ]:
def ask_database(sql_agent, question: str) -> str:
    """Send one natural-language question to the SQL agent."""

    if not question.strip():
        raise ValueError("Question cannot be empty.")
    return sql_agent.invoke({"input": question.strip()})["output"]

In [ ]:
print(ask_database(agent, "How many customers are in each country?"))

## 2. Create the demo database

The demo database contains customers, products, and orders.

In [2]:
database_path = agent_folder / "demo_notebook.sqlite"
database_path = create_demo_database(database_path)

print(f"Database created: {database_path.name}")

Database created: demo_notebook.sqlite


## 3. Build a read-only SQL agent

The production default is read-only, which protects the database from accidental changes.

In [4]:
agent, database = build_agent(database_path, read_only=True)

print("Connected tables:")
print(database.get_usable_table_names())

Connected tables:
['customers', 'orders', 'products']


## 4. Ask a question

Write a normal-language question. The agent inspects the schema, generates SQL, runs it, and formats the answer.

In [5]:
question = "What are the top 3 products by revenue?"
answer = ask_database(agent, question)

print(answer)

The top 3 products by revenue are:

1. **Laptop Pro** - $2599.98
2. **Standing Desk** - $599.99
3. **Wireless Mouse** - $149.95


## 5. Try another question

In [6]:
question = "How many customers are in each country?"
print(ask_database(agent, question))

The number of customers in each country is as follows:

- USA: 2 customers
- UK: 1 customer
- Brazil: 1 customer
